In [1]:
import os
import re
import sys
import json
import torch
import subprocess
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from tqdm import tqdm
from torchcrf import CRF

# --- Configuration ---
TRAIN_FILE = "data/train.jsonl"
VAL_FILE = "data/validation.jsonl"
TEST_FILE = "data/test.jsonl"
OUTPUT_DIR = "./bilstm_crf_modelV2"

BATCH_SIZE = 64
EPOCHS = 60 
LEARNING_RATE = 0.001 
WORD_EMBED_DIM = 300
CHAR_EMBED_DIM = 100
CHAR_CNN_FILTERS = 100
HIDDEN_DIM = 512
NUM_LSTM_LAYERS = 2      # <--- NEW: Add this

def get_emptiest_gpu_safely():
    if not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        print("\n🔍 Scanning available GPUs safely via nvidia-smi...")
        # Added utilization.gpu to the query
        result = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,memory.free,utilization.gpu", "--format=csv,nounits,noheader"], 
            encoding="utf-8"
        )
        
        best_id = -1
        max_free_mb = 0
        
        # Fallback variables in case ALL GPUs are >50% utilized
        fallback_id = 0
        fallback_max_mb = 0
        
        for line in result.strip().split("\n"):
            parts = line.split(", ")
            gpu_id = int(parts[0])
            free_memory = int(parts[1])
            gpu_util = int(parts[2])  # Extracted GPU utilization
            
            print(f"  GPU {gpu_id}: {free_memory} MB free | {gpu_util}% util")
            
            # Track the highest memory GPU globally as a fallback
            if free_memory > fallback_max_mb:
                fallback_max_mb = free_memory
                fallback_id = gpu_id
                
            # Primary logic: Only consider GPUs with < 50% utilization
            if gpu_util < 30:
                if free_memory > max_free_mb:
                    max_free_mb = free_memory
                    best_id = gpu_id
                    
        # Final Selection
        if best_id != -1:
            print(f"--> Selected GPU {best_id} with {max_free_mb} MB free VRAM and low utilization.\n")
            return torch.device(f"cuda:{best_id}")
        else:
            print(f"⚠️ All GPUs are highly utilized (>= 30%). Falling back to GPU {fallback_id} with {fallback_max_mb} MB free.\n")
            return torch.device(f"cuda:{fallback_id}")
            
    except Exception as e:
        print(f"⚠️ Failed to query nvidia-smi: {e}. Falling back to cuda:0.")
        return torch.device("cuda:0")

# --- 1. Data Alignment & Tokenization ---
def reconstruct_char_labels(input_text, output_dict):
    char_labels = ["O"] * len(input_text)
    
    field_to_tag = {
        "flat": "UNIT",
        "floor": "FLOOR",
        "block": "BLOCK",             # <-- ADD THIS
        "phase": "PHASE",             # <-- ADD THIS
        "building_name": "BUILDING_NAME",
        "estate_name": "ESTATE_NAME",
        "street_name": "STREET_NAME",
        "sub_district": "SUB_DISTRICT",
        "district": "DISTRICT",
        "region": "REGION",
        "village_name": "VILLAGE_NAME", 
        "building_number": "BUILDING_NUMBER"
    }
    
    # Flatten line1 and line2 into a single dictionary
    flat_targets = {}
    if "line1" in output_dict:
        flat_targets.update(output_dict.get("line1", {}))
        flat_targets.update(output_dict.get("line2", {}))
    else:
        flat_targets = output_dict

    # 1. Gather all terms to process
    items_to_process = []
    for field, tag in field_to_tag.items():
        val = flat_targets.get(field, "")
        if not val:
            continue
            
        search_terms = val.split(" / ") if " / " in val else [val]
        for term in search_terms:
            if term:
                items_to_process.append((field, tag, term))
                
    # 2. CRITICAL FIX: Sort by length of term descending. 
    # This ensures "MUI WO KAU TSUEN" is processed before "MUI WO"
    items_to_process.sort(key=lambda x: len(x[2]), reverse=True)
    
    # 3. Apply tags while avoiding overwriting
    for field, tag, term in items_to_process:
        start_idx = 0
        while True:
            idx = input_text.find(term, start_idx)
            if idx == -1:
                break # Term not found or no more occurrences
            
            # Check if this specific occurrence is already tagged by a longer entity
            is_already_tagged = any(char_labels[i] != "O" for i in range(idx, idx + len(term)))
            
            if not is_already_tagged:
                # Mark B- and I- tags on this specific untagged span
                char_labels[idx] = f"B-{tag}"
                for i in range(idx + 1, idx + len(term)):
                    if i < len(char_labels):
                        char_labels[i] = f"I-{tag}"
                break # Successfully tagged, move to the next term in items_to_process
            else:
                # This occurrence was already tagged, keep searching forward!
                start_idx = idx + 1
                
    return char_labels

def tokenize_and_align(input_text, output_dict):
    """Splits text into English words and Chinese chars, aligning the tags."""
    char_labels = reconstruct_char_labels(input_text, output_dict)
    tokens, token_tags = [], []
    
    # Regex: Matches English/Numbers natively, Chinese chars individually, and symbols
    # The updated regex
    for match in re.finditer(r'[a-zA-Z]+|[0-9]+|[\u4e00-\u9fff]|[^\s]', input_text):
        token_str = match.group()
        start_idx = match.start()
        
        tokens.append(token_str)
        # The tag for the entire word/token is based on its first character
        token_tags.append(char_labels[start_idx])
        
    return tokens, token_tags

# --- 2. Vocab Builder ---
class Vocab:
    def __init__(self):
        self.w2i = {"<PAD>": 0, "<UNK>": 1}
        self.i2w = {0: "<PAD>", 1: "<UNK>"}
        
    def add(self, word):
        if word not in self.w2i:
            idx = len(self.w2i)
            self.w2i[word] = idx
            self.i2w[idx] = word
            
    def __len__(self):
        return len(self.w2i)

# --- 3. PyTorch Dataset & Dataloader ---
class AddressDataset(Dataset):
    def __init__(self, data_list, word_vocab, char_vocab, tag2idx):
        self.data = data_list
        self.w2i = word_vocab
        self.c2i = char_vocab
        self.t2i = tag2idx
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        tokens, tags = self.data[idx]
        
        word_ids = [self.w2i.w2i.get(t, self.w2i.w2i["<UNK>"]) for t in tokens]
        char_ids_list = [[self.c2i.w2i.get(c, self.c2i.w2i["<UNK>"]) for c in token] for token in tokens]
        tag_ids = [self.t2i[tag] for tag in tags]
        
        return word_ids, char_ids_list, tag_ids

def collate_fn(batch):
    max_seq_len = max(len(item[0]) for item in batch)
    max_word_len = max([1] + [len(c) for item in batch for c in item[1]])
    
    b_words, b_chars, b_labels, b_masks = [], [], [], []
    
    for word_ids, char_ids_list, label_ids in batch:
        seq_len = len(word_ids)
        b_words.append(word_ids + [0] * (max_seq_len - seq_len))
        b_labels.append(label_ids + [0] * (max_seq_len - seq_len))
        b_masks.append([True] * seq_len + [False] * (max_seq_len - seq_len)) 
        
        padded_chars = [chars + [0] * (max_word_len - len(chars)) for chars in char_ids_list]
        padded_chars.extend([[0] * max_word_len for _ in range(max_seq_len - seq_len)])
        b_chars.append(padded_chars)
        
    return {
        "word_ids": torch.tensor(b_words, dtype=torch.long),
        "char_ids": torch.tensor(b_chars, dtype=torch.long),
        "labels": torch.tensor(b_labels, dtype=torch.long),
        "mask": torch.tensor(b_masks, dtype=torch.bool)
    }

# --- 4. The Neural Architecture ---
class BiLSTM_CNN_CRF(nn.Module):
    def __init__(self, vocab_size, char_vocab_size, num_tags, word_dim, char_dim, cnn_filters, hidden_dim, num_layers=1, dropout=0.5):
        super().__init__()
        
        # Word Embeddings
        self.word_embed = nn.Embedding(vocab_size, word_dim, padding_idx=0)
        
        # Character Embeddings & CNN
        self.char_embed = nn.Embedding(char_vocab_size, char_dim, padding_idx=0)
        # Padding=1 prevents convolution crashes on 1-character tokens (like Chinese chars)
        self.char_cnn = nn.Conv1d(in_channels=char_dim, out_channels=cnn_filters, kernel_size=3, padding=1)
        
        # BiLSTM
        lstm_input_dim = word_dim + cnn_filters
        self.lstm = nn.LSTM(lstm_input_dim, hidden_dim // 2, num_layers=num_layers, bidirectional=True, batch_first=True)
        
        # Projection & CRF (lexicon features removed from input dim)
        self.dropout = nn.Dropout(dropout)
        self.hidden2tag = nn.Linear(hidden_dim, num_tags)
        self.crf = CRF(num_tags, batch_first=True)
        
    def forward(self, word_ids, char_ids, mask, labels=None):
        batch_size, seq_len = word_ids.shape
        max_word_len = char_ids.shape[2]
        
        # 1. Word Vectors
        w_emb = self.word_embed(word_ids) # shape: (batch, seq, word_dim)
        
        # 2. Character Features (CNN)
        char_ids_flat = char_ids.view(-1, max_word_len) 
        c_emb = self.char_embed(char_ids_flat).permute(0, 2, 1) # reshape for PyTorch Conv1d
        
        c_cnn_out, _ = torch.max(self.char_cnn(c_emb), dim=2)   # Max pooling over the word
        c_features = c_cnn_out.view(batch_size, seq_len, -1)    # shape: (batch, seq, cnn_filters)
        
        # 3. Concatenate and run BiLSTM
        lstm_in = self.dropout(torch.cat([w_emb, c_features], dim=2))
        lstm_out, _ = self.lstm(lstm_in)
        
        # 4. Generate Logits directly from LSTM output
        emissions = self.hidden2tag(lstm_out)
        
        # 5. CRF Decoding / Loss
        if labels is not None:
            return -self.crf(emissions, tags=labels, mask=mask, reduction='mean')
        return self.crf.decode(emissions, mask=mask)

# --- 5. Main Training Pipeline ---
def main():
    device = get_emptiest_gpu_safely()
    
    tags = ["O"]
    tag_list = ["UNIT", "FLOOR", "BUILDING_NAME", "ESTATE_NAME", "STREET_NAME", 
                "SUB_DISTRICT", "DISTRICT", "REGION", "VILLAGE_NAME", "BUILDING_NUMBER",
                "BLOCK", "PHASE"]
                
    for t in tag_list:
        tags.extend([f"B-{t}", f"I-{t}"])
    tag2idx = {t: i for i, t in enumerate(tags)}
    
    word_vocab = Vocab()
    char_vocab = Vocab()
    
    def parse_file(file_path, update_vocab=False):
        data_list = []
        if not os.path.exists(file_path):
            return data_list
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                data = json.loads(line)
                tokens, token_tags = tokenize_and_align(data["input"], data["output"])
                if update_vocab:
                    for token in tokens:
                        word_vocab.add(token)
                        for char in token:
                            char_vocab.add(char)
                data_list.append((tokens, token_tags))
        return data_list

    print("📂 Parsing datasets and building Vocabularies...")
    train_data = parse_file(TRAIN_FILE, update_vocab=True)
    val_data = parse_file(VAL_FILE, update_vocab=False)

    if len(train_data) == 0:
        raise FileNotFoundError(f"🚨 No training data found! Please ensure '{TRAIN_FILE}' exists and is not empty.")

    print(f"✅ Vocab sizes -> Words: {len(word_vocab)}, Chars: {len(char_vocab)}")

    train_loader = DataLoader(AddressDataset(train_data, word_vocab, char_vocab, tag2idx), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(AddressDataset(val_data, word_vocab, char_vocab, tag2idx), batch_size=BATCH_SIZE*2, shuffle=False, collate_fn=collate_fn) if val_data else None

    print("⏳ Initializing NER Architecture...")
    model = BiLSTM_CNN_CRF(
        vocab_size=len(word_vocab), 
        char_vocab_size=len(char_vocab), 
        num_tags=len(tag2idx),
        word_dim=WORD_EMBED_DIM, 
        char_dim=CHAR_EMBED_DIM, 
        cnn_filters=CHAR_CNN_FILTERS, 
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LSTM_LAYERS 
    ).to(device)
    
    optimizer = Adam(model.parameters(), lr=LEARNING_RATE)
    
    # --- CHECKPOINT RESUME LOGIC ---
    start_epoch = 0
    best_val_loss = float('inf')
    checkpoint_path = os.path.join(OUTPUT_DIR, "checkpoint_latest.pt")
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    if os.path.exists(checkpoint_path):
        print(f"🔄 Found existing checkpoint. Resuming training from {checkpoint_path}...")
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint['best_val_loss']
        print(f"▶️ Resuming at Epoch {start_epoch + 1} with Best Val Loss: {best_val_loss:.4f}")

    print("\n🚀 Training Initiated...")
    
    try:
        for epoch in range(start_epoch, EPOCHS):
            model.train()
            total_train_loss = 0
            
            # --- TQDM FOR TRAINING ---
            train_iterator = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False, dynamic_ncols=True)
            
            for step, batch in enumerate(train_iterator):
                word_ids, char_ids = batch["word_ids"].to(device), batch["char_ids"].to(device)
                labels, mask = batch["labels"].to(device), batch["mask"].to(device)
                
                optimizer.zero_grad()
                loss = model(word_ids, char_ids, mask, labels=labels)
                loss.backward()
                optimizer.step()
                
                total_train_loss += loss.item()
                
                # Update the progress bar with the current loss
                train_iterator.set_postfix(loss=f"{loss.item():.4f}")
                
            avg_train_loss = total_train_loss / len(train_loader)

            # --- TQDM FOR VALIDATION ---
            if val_loader:
                model.eval()
                total_val_loss = 0
                val_iterator = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]  ", leave=False, dynamic_ncols=True)
                
                with torch.no_grad():
                    for batch in val_iterator:
                        word_ids, char_ids = batch["word_ids"].to(device), batch["char_ids"].to(device)
                        labels, mask = batch["labels"].to(device), batch["mask"].to(device)
                        val_loss = model(word_ids, char_ids, mask, labels=labels)
                        total_val_loss += val_loss.item()
                        val_iterator.set_postfix(loss=f"{val_loss.item():.4f}")
                
                avg_val_loss = total_val_loss / len(val_loader)
                
                # Print epoch summary
                print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

                # Save best model
                if avg_val_loss < best_val_loss:
                    best_val_loss = avg_val_loss
                    print(f"  🌟 New best validation loss! Saving best model...")
                    torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "pytorch_model.bin"))
                    with open(os.path.join(OUTPUT_DIR, "vocabs.json"), "w", encoding="utf-8") as f:
                        json.dump({"w2i": word_vocab.w2i, "c2i": char_vocab.w2i, "t2i": tag2idx}, f)
            else:
                print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f}")

            # --- SAVE CHECKPOINT AFTER EVERY EPOCH ---
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss': best_val_loss,
            }, checkpoint_path)

        print(f"\n✅ Training complete. Best model saved to {OUTPUT_DIR}")

    except KeyboardInterrupt:
        print("\n\n⚠️ Training Interrupted by User (Ctrl+C)!")
        
        # Save a manual checkpoint on interrupt so progress isn't lost
        interrupt_path = os.path.join(OUTPUT_DIR, "checkpoint_interrupted.pt")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss': best_val_loss,
        }, interrupt_path)
        print(f"💾 Emergency checkpoint saved to {interrupt_path}")
        print("👋 Safe exit complete.")
        sys.exit(0)

if __name__ == "__main__":
    main()


🔍 Scanning available GPUs safely via nvidia-smi...
  GPU 0: 373 MB free | 100% util
  GPU 1: 7290 MB free | 0% util
  GPU 2: 22904 MB free | 0% util
  GPU 3: 22904 MB free | 0% util
--> Selected GPU 2 with 22904 MB free VRAM and low utilization.

📂 Parsing datasets and building Vocabularies...
✅ Vocab sizes -> Words: 26745, Chars: 3031
⏳ Initializing NER Architecture...
🔄 Found existing checkpoint. Resuming training from ./bilstm_crf_modelV2/checkpoint_latest.pt...
▶️ Resuming at Epoch 8 with Best Val Loss: 0.3072

🚀 Training Initiated...


Epoch 8/60 | Train Loss: 0.2571 | Val Loss: 0.3056
  🌟 New best validation loss! Saving best model...


Epoch 9/60 | Train Loss: 0.2456 | Val Loss: 0.2920
  🌟 New best validation loss! Saving best model...


Epoch 10/60 | Train Loss: 0.2393 | Val Loss: 0.2903
  🌟 New best validation loss! Saving best model...


Epoch 11/60 | Train Loss: 0.2301 | Val Loss: 0.2829
  🌟 New best validation loss! Saving best model...




⚠️ Training Interrupted by User (Ctrl+C)!
💾 Emergency checkpoint saved to ./bilstm_crf_modelV2/checkpoint_interrupted.pt
👋 Safe exit complete.


SystemExit: 0

/usr/local/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3680: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [2]:
#!/usr/bin/env python3
"""
Batch Evaluation Script for BiLSTM-CNN-CRF
Includes Multi-Threshold Confidence Diagnostics (Baseline, 50%, 60%, 70%, 80%).
Features: Calibrated Accuracy, Situation-Aware Confidence, 
Combined Floor+Flat Evaluation, and Space-Insensitive Matching.
"""

import os
import re
import json
import time
import torch
import subprocess
import torch.nn as nn
from collections import defaultdict
from tqdm.auto import tqdm
from torchcrf import CRF

# ==========================================
# CONFIGURATION (Match your training params)
# ==========================================
MODEL_DIR = "./bilstm_crf_modelV2"
LOG_FILE = "parsing_results_bilstm.log"
TEST_FILE = "data/test.jsonl"
BATCH_SIZE = 64

WORD_EMBED_DIM = 300
CHAR_EMBED_DIM = 100
CHAR_CNN_FILTERS = 100
HIDDEN_DIM = 512
NUM_LSTM_LAYERS = 2 

EXCLUDE_FROM_OVERALL = {"DISTRICT", "REGION", "SUB_DISTRICT"}
THRESHOLDS = [0.0, 0.50, 0.60, 0.70, 0.80]

# ==========================================
# MODEL ARCHITECTURE (Modified for eval)
# ==========================================
class BiLSTM_CNN_CRF(nn.Module):
    def __init__(self, vocab_size, char_vocab_size, num_tags, word_dim, char_dim, cnn_filters, hidden_dim, num_layers=2, dropout=0.5):
        super().__init__()
        self.word_embed = nn.Embedding(vocab_size, word_dim, padding_idx=0)
        self.char_embed = nn.Embedding(char_vocab_size, char_dim, padding_idx=0)
        self.char_cnn = nn.Conv1d(in_channels=char_dim, out_channels=cnn_filters, kernel_size=3, padding=1)
        
        lstm_input_dim = word_dim + cnn_filters
        self.lstm = nn.LSTM(lstm_input_dim, hidden_dim // 2, num_layers=num_layers, bidirectional=True, batch_first=True)
        
        self.dropout = nn.Dropout(dropout)
        self.hidden2tag = nn.Linear(hidden_dim, num_tags)
        self.crf = CRF(num_tags, batch_first=True)
        
    def forward(self, word_ids, char_ids, mask, labels=None):
        batch_size, seq_len = word_ids.shape
        max_word_len = char_ids.shape[2]
        
        w_emb = self.word_embed(word_ids) 
        
        char_ids_flat = char_ids.view(-1, max_word_len) 
        c_emb = self.char_embed(char_ids_flat).permute(0, 2, 1)
        
        c_cnn_out, _ = torch.max(self.char_cnn(c_emb), dim=2)   
        c_features = c_cnn_out.view(batch_size, seq_len, -1)    
        
        lstm_in = self.dropout(torch.cat([w_emb, c_features], dim=2))
        lstm_out, _ = self.lstm(lstm_in)
        
        emissions = self.hidden2tag(lstm_out)
        
        if labels is not None:
            return -self.crf(emissions, tags=labels, mask=mask, reduction='mean')
            
        # MODIFIED: Return tags AND emissions to calculate confidence probabilities
        tags = self.crf.decode(emissions, mask=mask)
        return tags, emissions

# ==========================================
# UTILITY FUNCTIONS
# ==========================================
def get_emptiest_gpu_safely():
    if not torch.cuda.is_available(): return -1
    try:
        result = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,memory.free", "--format=csv,nounits,noheader"], 
            encoding="utf-8"
        )
        best_id, max_free_mb = 0, -1
        for line in result.strip().split("\n"):
            if not line.strip(): continue
            gpu_id, free_memory = map(int, line.split(", "))
            if free_memory > max_free_mb:
                max_free_mb, best_id = free_memory, gpu_id
        return best_id
    except Exception: return 0

def tokenize_text(input_text):
    """Tokenize exactly as done in training."""
    tokens = []
    for match in re.finditer(r'[a-zA-Z]+|[0-9]+|[\u4e00-\u9fff]|[^\s]', input_text):
        tokens.append(match.group())
    return tokens

def prepare_batch(batch_texts, w2i, c2i):
    batch_tokens = [tokenize_text(text) for text in batch_texts]
    max_seq_len = max(len(t) for t in batch_tokens)
    max_word_len = max([1] + [len(c) for t in batch_tokens for c in t])
    
    b_words, b_chars, b_masks = [], [], []
    for tokens in batch_tokens:
        seq_len = len(tokens)
        word_ids = [w2i.get(t, w2i.get("<UNK>", 1)) for t in tokens]
        char_ids_list = [[c2i.get(c, c2i.get("<UNK>", 1)) for c in token] for token in tokens]
        
        b_words.append(word_ids + [0] * (max_seq_len - seq_len))
        b_masks.append([True] * seq_len + [False] * (max_seq_len - seq_len))
        
        padded_chars = [chars + [0] * (max_word_len - len(chars)) for chars in char_ids_list]
        padded_chars.extend([[0] * max_word_len for _ in range(max_seq_len - seq_len)])
        b_chars.append(padded_chars)
        
    return (
        torch.tensor(b_words, dtype=torch.long),
        torch.tensor(b_chars, dtype=torch.long),
        torch.tensor(b_masks, dtype=torch.bool),
        batch_tokens
    )

def extract_3d_components(parsed_entities):
    components = defaultdict(list)
    confs = defaultdict(list)

    for entity in parsed_entities:
        tag = entity["entity_group"]
        if tag == "O": continue
        word = entity["word"].strip()
        if word:
            components[tag].append(word)
            confs[tag].append(entity["conf"])

    formatted_output, conf_output = {}, {}
    for tag, words in components.items():
        joined_string = "".join(words)
        if any("\u4e00" <= char <= "\u9fff" for char in joined_string):
            formatted_output[tag] = "".join(words)
        else:
            joined_en = " ".join(words).strip()
            formatted_output[tag] = re.sub(r"\s*([/\.-])\s*", r"\1", joined_en)

        conf_output[tag] = sum(confs[tag]) / len(confs[tag]) if confs[tag] else 0.0

    return formatted_output, conf_output

def assemble_compact_json(extracted_data):
    return {
        "line1": {
            "flat": extracted_data.get("UNIT", ""), "floor": extracted_data.get("FLOOR", ""),
            "block": extracted_data.get("BLOCK", ""), "phase": extracted_data.get("PHASE", ""),
            "building_name": extracted_data.get("BUILDING_NAME", ""),
        },
        "line2": {
            "estate_name": extracted_data.get("ESTATE_NAME", ""), "village_name": extracted_data.get("VILLAGE_NAME", ""),        
            "building_number": extracted_data.get("BUILDING_NUMBER", ""), "street_name": extracted_data.get("STREET_NAME", ""),
            "sub_district": extracted_data.get("SUB_DISTRICT", ""), "district": extracted_data.get("DISTRICT", ""),
            "region": extracted_data.get("REGION", ""),
        }
    }

def flatten_json(output_dict):
    flat = {}
    if "line1" in output_dict:
        flat.update(output_dict.get("line1", {}))
        flat.update(output_dict.get("line2", {}))
    else:
        flat = output_dict
    return {k: v for k, v in flat.items() if v}

def compute_situation_aware_confidence(conf_mapped):
    core = set()
    for key in ["UNIT", "FLOOR", "BLOCK", "PHASE", "ESTATE_NAME", "BUILDING_NAME", "VILLAGE_NAME", "STREET_NAME", "BUILDING_NUMBER"]:
        if key in conf_mapped: core.add(key)

    overall = 1.0
    used = []
    
    for k in core:
        if k in conf_mapped and k not in EXCLUDE_FROM_OVERALL and conf_mapped[k] > 0:
            overall *= conf_mapped[k]
            used.append(k)

    if not used:
        for k, c in conf_mapped.items():
            if k not in EXCLUDE_FROM_OVERALL and c > 0:
                overall *= c
                used.append(k)
    return overall

def normalize_for_eval(text):
    if not text: return ""
    return re.sub(r'\s+', '', str(text).lower())

# ==========================================
# MAIN EXECUTION
# ==========================================
def main():
    device_id = get_emptiest_gpu_safely()
    device = torch.device(f"cuda:{device_id}" if device_id != -1 else "cpu")
    
    print(f"DEBUG: Loading Vocabs and Model from {MODEL_DIR}...")
    vocab_path = os.path.join(MODEL_DIR, "vocabs.json")
    weights_path = os.path.join(MODEL_DIR, "pytorch_model.bin")
    
    if not os.path.exists(vocab_path) or not os.path.exists(weights_path):
        print(f"❌ Error: Model or vocabs not found in {MODEL_DIR}.")
        return

    # Load Vocabs
    with open(vocab_path, "r", encoding="utf-8") as f:
        vocabs = json.load(f)
    w2i = vocabs["w2i"]
    c2i = vocabs["c2i"]
    t2i = vocabs["t2i"]
    idx2tag = {int(v): k for k, v in t2i.items()}
    
    # Initialize Model
    model = BiLSTM_CNN_CRF(
        vocab_size=len(w2i), 
        char_vocab_size=len(c2i), 
        num_tags=len(t2i),
        word_dim=WORD_EMBED_DIM, 
        char_dim=CHAR_EMBED_DIM, 
        cnn_filters=CHAR_CNN_FILTERS, 
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LSTM_LAYERS
    )
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.to(device)
    model.eval()

    all_fields = [
        "flat", "floor", "building_name", "block", "phase",
        "estate_name", "village_name", "building_number",
        "street_name", "sub_district", "district", "region"
    ]
    
    tag_map = {
        "flat": "UNIT", "floor": "FLOOR", "block": "BLOCK",
        "phase": "PHASE", "building_name": "BUILDING_NAME",
        "estate_name": "ESTATE_NAME", "village_name": "VILLAGE_NAME",
        "building_number": "BUILDING_NUMBER", "street_name": "STREET_NAME",
        "sub_district": "SUB_DISTRICT", "district": "DISTRICT", "region": "REGION",
    }
    
    total_time = 0.0
    stats = {t: {f: {'TP': 0, 'FP': 0, 'FN': 0} for f in all_fields} for t in THRESHOLDS}
    exact_matches = {t: 0 for t in THRESHOLDS}
    total_in_bin = {t: 0 for t in THRESHOLDS}

    print("🚀 Running batch evaluation...")
    with open(TEST_FILE, "r", encoding="utf-8") as file:
        raw_lines = [line.strip() for line in file if line.strip() and not line.startswith('#')]
    
    test_data = [json.loads(line) for line in raw_lines]
    
    with open(LOG_FILE, "w", encoding="utf-8") as log:
        for i in tqdm(range(0, len(test_data), BATCH_SIZE), desc="Batches"):
            batch = test_data[i:i + BATCH_SIZE]
            batch_texts = [item["input"].strip() for item in batch]
            
            start_time = time.perf_counter()
            
            # Prepare tensors via custom tokenization
            word_ids, char_ids, masks, batch_tokens = prepare_batch(batch_texts, w2i, c2i)
            word_ids = word_ids.to(device)
            char_ids = char_ids.to(device)
            masks = masks.to(device)
            
            with torch.no_grad():
                batch_predictions, batch_emissions = model(word_ids, char_ids, masks)
                batch_probs = torch.softmax(batch_emissions, dim=-1).cpu()

            total_time += time.perf_counter() - start_time

            for b_idx, item in enumerate(batch):
                address = batch_texts[b_idx]
                ground_truth_flat = flatten_json(item.get("output", {}))
                
                prediction_ids = batch_predictions[b_idx]
                item_probs = batch_probs[b_idx]
                tokens = batch_tokens[b_idx]
                
                parsed_entities = []
                # Map token IDs back to strings and collect probabilities
                for idx, tag_id in enumerate(prediction_ids):
                    token_str = tokens[idx]
                    tag = idx2tag[tag_id]
                    conf = item_probs[idx, tag_id].item()
                    entity_group = tag.replace("B-", "").replace("I-", "") if tag != "O" else "O"
                    
                    parsed_entities.append({
                        "entity_group": entity_group, "word": token_str, "conf": conf,
                    })

                extracted_data, conf_output = extract_3d_components(parsed_entities)
                predicted_flat = flatten_json(assemble_compact_json(extracted_data))
                
                overall_conf = compute_situation_aware_confidence(conf_output)
                
                # --- STATS CALCULATION FOR FULL PREDICTION ---
                pred_floor_flat = normalize_for_eval(predicted_flat.get("floor", "") + predicted_flat.get("flat", ""))
                gt_floor_flat = normalize_for_eval(ground_truth_flat.get("floor", "") + ground_truth_flat.get("flat", ""))
                
                is_perfect_match = True
                field_evals = {} 
                
                for field in all_fields:
                    pred_val = predicted_flat.get(field, "")
                    gt_val = ground_truth_flat.get(field, "")
                    
                    pred_norm = normalize_for_eval(pred_val)
                    gt_norm = normalize_for_eval(gt_val)
                    
                    is_correct = False
                    
                    if pred_norm == gt_norm:
                        is_correct = True
                    elif field in ['floor', 'flat'] and pred_floor_flat == gt_floor_flat and pred_floor_flat != "":
                        is_correct = True
                        
                    if not is_correct:
                        is_perfect_match = False
                        
                    # Standard Confusion Matrix variables
                    tp = fp = fn = 0
                    if pred_norm or gt_norm:
                        if is_correct:
                            tp = 1
                        else:
                            if pred_norm and gt_norm:
                                fp = 1; fn = 1
                            elif pred_norm and not gt_norm:
                                fp = 1
                            elif not pred_norm and gt_norm:
                                fn = 1
                    
                    field_evals[field] = (tp, fp, fn)

                # Assign these stats to all bins where overall_conf met the threshold
                for t in THRESHOLDS:
                    if overall_conf >= t:
                        total_in_bin[t] += 1
                        if is_perfect_match:
                            exact_matches[t] += 1
                        
                        for field in all_fields:
                            tp, fp, fn = field_evals[field]
                            stats[t][field]['TP'] += tp
                            stats[t][field]['FP'] += fp
                            stats[t][field]['FN'] += fn

                # --- WRITING LOGS ---
                log.write(f"Original: {address}\n")
                
                if is_perfect_match:
                    log.write("✅ EXACT MATCH\n")
                else:
                    log.write("❌ MISMATCH FOUND\n")
                
                for field in all_fields:
                    pred_val = predicted_flat.get(field, "")
                    gt_val = ground_truth_flat.get(field, "")
                    
                    if pred_val or gt_val:
                        tp, fp, fn = field_evals[field]
                        status = "✅" if tp == 1 else "❌"
                        
                        tag = tag_map.get(field, field.upper())
                        conf_str = f"  conf={conf_output.get(tag, 0.0):.4f}" if tag in conf_output else ""
                        
                        log.write(f" {status} {field.upper()}:{conf_str}\n")
                        log.write(f"   PRED: {pred_val if pred_val else '[None]'}\n")
                        log.write(f"   TRUE: {gt_val if gt_val else '[None]'}\n")

                log.write(f"Per-label confidences : { {k: round(v, 4) for k, v in conf_output.items()} }\n")
                log.write(f"Overall confidence    : {overall_conf:.6f}\n")
                log.write("-" * 50 + "\n")

    print("\n" + "=" * 65)
    print("📊 CALIBRATED METRICS EVALUATION")
    print("=" * 65)
    print(f"Total Addresses Tested: {len(test_data)}")
    print(f"⏱️ Total Inference runtime: {total_time:.4f} seconds\n")

    for t in THRESHOLDS:
        title = "ALL PREDICTIONS" if t == 0.0 else f"PREDICTIONS WITH >= {int(t*100)}% CONFIDENCE"
        print("=" * 65)
        print(f"🚀 {title}")
        print("=" * 65)
        
        bin_count = total_in_bin[t]
        if bin_count > 0:
            exact_match_acc = (exact_matches[t] / bin_count) * 100
            print(f"Whole-Address Perfect Match : {exact_match_acc:.2f}% ({exact_matches[t]}/{bin_count})\n")
            
            print(f"{'FIELD':<16} | {'PRECISION':<9} | {'RECALL':<9} | {'F1-SCORE':<9} | {'SUPPORT'}")
            print("-" * 65)
            
            macro_f1 = 0
            valid_fields = 0
            
            for field in all_fields:
                tp = stats[t][field]['TP']
                fp = stats[t][field]['FP']
                fn = stats[t][field]['FN']
                
                support = tp + fn
                
                precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
                recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
                f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
                
                if support > 0:
                    macro_f1 += f1
                    valid_fields += 1
                    
                p_str = f"{precision*100:>5.1f}%"
                r_str = f"{recall*100:>5.1f}%"
                f1_str = f"{f1*100:>5.1f}%"
                
                print(f"{field:<16} | {p_str:<9} | {r_str:<9} | {f1_str:<9} | {support}")

            if valid_fields > 0:
                print("-" * 65)
                print(f"{'MACRO AVERAGE':<16} | {'-':<9} | {'-':<9} | {(macro_f1/valid_fields)*100:>5.1f}%  |")
        else:
            print(f"No samples met the >= {int(t*100)}% confidence threshold.")
        print("\n")

    print(f"✅ Processing complete. Raw baseline logs saved to {LOG_FILE}")

if __name__ == "__main__":
    main()

DEBUG: Loading Vocabs and Model from ./bilstm_crf_modelV2...
🚀 Running batch evaluation...


Batches:   0%|          | 0/324 [00:00<?, ?it/s]


📊 CALIBRATED METRICS EVALUATION
Total Addresses Tested: 20696
⏱️ Total Inference runtime: 16.6702 seconds

🚀 ALL PREDICTIONS
Whole-Address Perfect Match : 89.48% (18519/20696)

FIELD            | PRECISION | RECALL    | F1-SCORE  | SUPPORT
-----------------------------------------------------------------
flat             |  99.2%    |  98.7%    |  98.9%    | 14380
floor            |  99.7%    |  99.0%    |  99.4%    | 18507
building_name    |  97.7%    |  97.7%    |  97.7%    | 11435
block            |  98.9%    |  96.3%    |  97.6%    | 2772
phase            |  97.7%    |  94.5%    |  96.1%    | 586
estate_name      |  96.5%    |  95.6%    |  96.1%    | 5224
village_name     |  98.8%    |  86.9%    |  92.5%    | 6862
building_number  |  99.7%    |  99.2%    |  99.5%    | 20215
street_name      |  99.7%    |  99.8%    |  99.7%    | 13579
sub_district     |  99.5%    |  95.7%    |  97.6%    | 18597
district         |  99.9%    |  95.4%    |  97.6%    | 20410
region           |  98.3%  

In [4]:
#!/usr/bin/env python3
"""
HK Address Parser - Final App with Integrated Evaluation (BiLSTM-CNN-CRF Version)
Evaluates address splitting logic, calculates split-aware confidence, 
and measures Accuracy relative to Confidence Thresholds.
Includes interchangeable logic for Estate Name and Building Name.
"""

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import re
import json
import time
import torch
import subprocess
import torch.nn as nn
from collections import defaultdict
from tqdm.auto import tqdm
from torchcrf import CRF

# ==========================================
# CONFIGURATION
# ==========================================
MODEL_DIR = "./bilstm_crf_modelV2"
LOG_FILE = "address_split_results_bilstmV2.log"
TEST_FILE = "data/test.jsonl"
BATCH_SIZE = 32

WORD_EMBED_DIM = 300
CHAR_EMBED_DIM = 100
CHAR_CNN_FILTERS = 100
HIDDEN_DIM = 512
NUM_LSTM_LAYERS = 2 

THRESHOLDS = [0.0, 0.2, 0.3, 0.4, 0.50, 0.60, 0.70, 0.80]
ALL_FIELDS = [
    "flat", "floor", "building_name", "block", "phase",
    "estate_name", "village_name", "building_number",
    "street_name", "sub_district", "district", "region"
]

# ==========================================
# UTILITY FUNCTIONS FOR EVALUATION
# ==========================================
def flatten_json(output_dict):
    """Flattens the nested ground truth JSON from test.jsonl."""
    flat = {}
    if "line1" in output_dict and isinstance(output_dict["line1"], dict):
        flat.update(output_dict.get("line1", {}))
        flat.update(output_dict.get("line2", {}))
    else:
        flat = output_dict
    return {k: v for k, v in flat.items() if v}

def normalize_for_eval(text):
    """Lowercases and removes ALL spaces/punctuation for strict string comparison."""
    if not text:
        return ""
    return re.sub(r'[\s,/\\\-;\.，。、；]+', '', str(text).lower())

def tokenize_text(input_text):
    """Tokenize exactly as done in BiLSTM training."""
    tokens = []
    for match in re.finditer(r'[a-zA-Z]+|[0-9]+|[\u4e00-\u9fff]|[^\s]', input_text):
        tokens.append(match.group())
    return tokens

def prepare_batch(batch_texts, w2i, c2i):
    """Pads and prepares tensors for the BiLSTM model."""
    batch_tokens = [tokenize_text(text) for text in batch_texts]
    max_seq_len = max(len(t) for t in batch_tokens)
    max_word_len = max([1] + [len(c) for t in batch_tokens for c in t])
    
    b_words, b_chars, b_masks = [], [], []
    for tokens in batch_tokens:
        seq_len = len(tokens)
        word_ids = [w2i.get(t, w2i.get("<UNK>", 1)) for t in tokens]
        char_ids_list = [[c2i.get(c, c2i.get("<UNK>", 1)) for c in token] for token in tokens]
        
        b_words.append(word_ids + [0] * (max_seq_len - seq_len))
        b_masks.append([True] * seq_len + [False] * (max_seq_len - seq_len))
        
        padded_chars = [chars + [0] * (max_word_len - len(chars)) for chars in char_ids_list]
        padded_chars.extend([[0] * max_word_len for _ in range(max_seq_len - seq_len)])
        b_chars.append(padded_chars)
        
    return (
        torch.tensor(b_words, dtype=torch.long),
        torch.tensor(b_chars, dtype=torch.long),
        torch.tensor(b_masks, dtype=torch.bool),
        batch_tokens
    )

# ==========================================
# CUSTOM ARCHITECTURE
# ==========================================
class BiLSTM_CNN_CRF(nn.Module):
    def __init__(self, vocab_size, char_vocab_size, num_tags, word_dim, char_dim, cnn_filters, hidden_dim, num_layers=2, dropout=0.5):
        super().__init__()
        self.word_embed = nn.Embedding(vocab_size, word_dim, padding_idx=0)
        self.char_embed = nn.Embedding(char_vocab_size, char_dim, padding_idx=0)
        self.char_cnn = nn.Conv1d(in_channels=char_dim, out_channels=cnn_filters, kernel_size=3, padding=1)
        
        lstm_input_dim = word_dim + cnn_filters
        self.lstm = nn.LSTM(lstm_input_dim, hidden_dim // 2, num_layers=num_layers, bidirectional=True, batch_first=True)
        
        self.dropout = nn.Dropout(dropout)
        self.hidden2tag = nn.Linear(hidden_dim, num_tags)
        self.crf = CRF(num_tags, batch_first=True)
        
    def forward(self, word_ids, char_ids, mask, labels=None):
        batch_size, seq_len = word_ids.shape
        max_word_len = char_ids.shape[2]
        
        w_emb = self.word_embed(word_ids) 
        
        char_ids_flat = char_ids.view(-1, max_word_len) 
        c_emb = self.char_embed(char_ids_flat).permute(0, 2, 1)
        
        c_cnn_out, _ = torch.max(self.char_cnn(c_emb), dim=2)   
        c_features = c_cnn_out.view(batch_size, seq_len, -1)    
        
        lstm_in = self.dropout(torch.cat([w_emb, c_features], dim=2))
        lstm_out, _ = self.lstm(lstm_in)
        
        emissions = self.hidden2tag(lstm_out)
        
        if labels is not None:
            return -self.crf(emissions, tags=labels, mask=mask, reduction='mean')
            
        tags = self.crf.decode(emissions, mask=mask)
        return tags, emissions

# ==========================================
# ADDRESS PARSER CLASS (BiLSTM Version)
# ==========================================
class HKAddressParserBiLSTM:
    def __init__(self, model_path):
        self.model_path = model_path
        self.device = self._get_emptiest_gpu_safely()
        print(f"DEBUG: Using Device -> {self.device}")

        vocab_path = os.path.join(model_path, "vocabs.json")
        weights_path = os.path.join(model_path, "pytorch_model.bin")
        
        if not os.path.exists(vocab_path) or not os.path.exists(weights_path):
            raise FileNotFoundError(f"Model or vocabs not found in {model_path}.")

        # Load Vocabs
        with open(vocab_path, "r", encoding="utf-8") as f:
            vocabs = json.load(f)
        self.w2i = vocabs["w2i"]
        self.c2i = vocabs["c2i"]
        self.t2i = vocabs["t2i"]
        self.idx2tag = {int(v): k for k, v in self.t2i.items()}
        
        # Initialize Model
        self.model = BiLSTM_CNN_CRF(
            vocab_size=len(self.w2i), 
            char_vocab_size=len(self.c2i), 
            num_tags=len(self.t2i),
            word_dim=WORD_EMBED_DIM, 
            char_dim=CHAR_EMBED_DIM, 
            cnn_filters=CHAR_CNN_FILTERS, 
            hidden_dim=HIDDEN_DIM,
            num_layers=NUM_LSTM_LAYERS
        )
        
        print(f"📦 Loading weights from {weights_path} ...")
        self.model.load_state_dict(torch.load(weights_path, map_location=self.device))
        self.model.to(self.device)
        self.model.eval()

    @staticmethod
    def _get_emptiest_gpu_safely():
        if not torch.cuda.is_available(): return torch.device("cpu")
        try:
            result = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=index,memory.free,utilization.gpu", "--format=csv,nounits,noheader"],
                encoding="utf-8"
            )
            best_id, max_free_mb = 0, 0
            for line in result.strip().split("\n"):
                parts = line.split(", ")
                gpu_id, free_memory, gpu_util = int(parts[0]), int(parts[1]), int(parts[2])
                if gpu_util < 30 and free_memory > max_free_mb:
                    max_free_mb, best_id = free_memory, gpu_id
            return torch.device(f"cuda:{best_id}")
        except Exception: return torch.device("cuda:0")

    @staticmethod
    def _extract_3d_components(parsed_entities):
        components = defaultdict(list)
        confs = defaultdict(list)

        for entity in parsed_entities:
            tag = entity["entity_group"]
            if tag == "O": continue
            components[tag].append(entity["word"])
            confs[tag].append(entity["conf"])

        formatted_output, conf_output = {}, {}
        for tag, words in components.items():
            formatted_output[tag] = "".join(words)
            conf_output[tag] = sum(confs[tag]) / len(confs[tag]) if confs[tag] else 0.0

        return formatted_output, conf_output

    @staticmethod
    def _map_extracted_to_standard_keys(extracted_data, conf_data=None):
        mapping = {
            "UNIT": "flat", "FLOOR": "floor", "BLOCK": "block", "PHASE": "phase",
            "BUILDING_NAME": "building_name", "ESTATE_NAME": "estate_name",
            "VILLAGE_NAME": "village_name", "BUILDING_NUMBER": "building_number",
            "STREET_NAME": "street_name", "SUB_DISTRICT": "sub_district",
            "DISTRICT": "district", "REGION": "region"
        }
        mapped = {mapping.get(k, k.lower()): v for k, v in extracted_data.items()}
        if conf_data is not None:
            mapped_conf = {mapping.get(k, k.lower()): v for k, v in conf_data.items()}
            return mapped, mapped_conf
        return mapped

    @staticmethod
    def _split_address(extracted_labels, original_input, parsed_entities, conf_mapped):
        """
        Splits address and calculates confidence strictly based on labels 
        that influenced the split decision. Includes logic to correctly bind 
        Building Numbers with Village and Street names, and separates Estate/Phase 
        from Building/Block when both hierarchical levels exist.
        """
        line1_keys = set()
        logic_keys_used = set()
        
        # 1. Micro elements
        if "flat" in extracted_labels: 
            line1_keys.add("flat")
            logic_keys_used.add("flat")
        if "floor" in extracted_labels: 
            line1_keys.add("floor")
            logic_keys_used.add("floor")

        # 2. Structural elements (Hierarchical separation logic)
        has_bldg = "building_name" in extracted_labels
        has_block = "block" in extracted_labels
        has_est = "estate_name" in extracted_labels
        has_phase = "phase" in extracted_labels
        has_vill = "village_name" in extracted_labels
        has_street = "street_name" in extracted_labels
        has_bldg_no = "building_number" in extracted_labels
        
        # RULE 1: If Building OR Block exists, they take priority for Line 1 (micro).
        if has_bldg or has_block:
            if has_bldg:
                line1_keys.add("building_name")
                logic_keys_used.add("building_name")
            if has_block:
                line1_keys.add("block")
                logic_keys_used.add("block")
                
        # RULE 2: If NO Building or Block, but Estate exists, Estate acts as the building (Line 1).
        elif has_est:
            line1_keys.add("estate_name")
            logic_keys_used.add("estate_name")
            if has_phase: 
                line1_keys.add("phase")
                logic_keys_used.add("phase")
                
        # RULE 3: Fallbacks for Village / Street / Building Number
        elif has_vill:
            line1_keys.add("village_name")
            logic_keys_used.add("village_name")
            if has_bldg_no: 
                line1_keys.add("building_number")
                logic_keys_used.add("building_number")
        elif has_street:
            line1_keys.add("street_name")
            logic_keys_used.add("street_name")
            if has_bldg_no: 
                line1_keys.add("building_number")
                logic_keys_used.add("building_number")
        elif has_bldg_no:
            line1_keys.add("building_number")
            logic_keys_used.add("building_number")

        # 3. Assign tokens
        is_chinese = any("\u4e00" <= char <= "\u9fff" for char in original_input)
        token_groups = []
        mapping = {
            "UNIT": "flat", "FLOOR": "floor", "BLOCK": "block", "PHASE": "phase",
            "BUILDING_NAME": "building_name", "ESTATE_NAME": "estate_name",
            "VILLAGE_NAME": "village_name", "BUILDING_NUMBER": "building_number",
            "STREET_NAME": "street_name", "SUB_DISTRICT": "sub_district",
            "DISTRICT": "district", "REGION": "region"
        }
        
        for entity in parsed_entities:
            raw_tag = entity["entity_group"]
            mapped_tag = mapping.get(raw_tag, raw_tag.lower())
            if mapped_tag in line1_keys: token_groups.append("micro")
            elif mapped_tag != "o" and mapped_tag != "O": token_groups.append("macro")
            else: token_groups.append("O") 
                
        # Resolve punctuation
        resolved_groups = []
        last_valid = "micro" if not is_chinese else "macro" 
        for tg in token_groups:
            if tg != "O":
                last_valid = tg
                resolved_groups.append(tg)
            else:
                resolved_groups.append(last_valid)
                
        # Group adjacent tokens into segments by their tag
        micro_segments, macro_segments = [], []
        curr_micro_seg, curr_micro_tag = [], None
        curr_macro_seg, curr_macro_tag = [], None
        
        for entity, group in zip(parsed_entities, resolved_groups):
            tag = mapping.get(entity["entity_group"], entity["entity_group"].lower())
            word = entity["word"]
            
            if group == "micro":
                if tag != 'o' and tag != 'O':
                    if curr_micro_tag != tag:
                        if curr_micro_seg:
                            micro_segments.append((curr_micro_tag, curr_micro_seg))
                        curr_micro_seg = [word]
                        curr_micro_tag = tag
                    else:
                        curr_micro_seg.append(word)
                else:
                    if curr_micro_seg: curr_micro_seg.append(word)
                    else: curr_micro_seg, curr_micro_tag = [word], "o"
            else:
                if tag != 'o' and tag != 'O':
                    if curr_macro_tag != tag:
                        if curr_macro_seg:
                            macro_segments.append((curr_macro_tag, curr_macro_seg))
                        curr_macro_seg = [word]
                        curr_macro_tag = tag
                    else:
                        curr_macro_seg.append(word)
                else:
                    if curr_macro_seg: curr_macro_seg.append(word)
                    else: curr_macro_seg, curr_macro_tag = [word], "o"
                        
        if curr_micro_seg: micro_segments.append((curr_micro_tag, curr_micro_seg))
        if curr_macro_seg: macro_segments.append((curr_macro_tag, curr_macro_seg))
            
        def reorder_segments(segments, is_chinese):
            """Moves the building_number chunk to sit directly next to the primary grouping element"""
            bldg_no_idx = next((i for i, s in enumerate(segments) if s[0] == "building_number"), -1)
            if bldg_no_idx == -1: return segments
            
            bldg_no_seg = segments.pop(bldg_no_idx)
            target_tags = ["village_name", "street_name", "estate_name", "building_name"]
            
            if is_chinese:
                target_idx = -1
                for i, s in enumerate(segments):
                    if s[0] in target_tags: target_idx = i
                if target_idx != -1: segments.insert(target_idx + 1, bldg_no_seg)
                else: segments.insert(0, bldg_no_seg)
            else:
                target_idx = -1
                for i, s in enumerate(segments):
                    if s[0] in target_tags:
                        target_idx = i
                        break
                if target_idx != -1: segments.insert(target_idx, bldg_no_seg)
                else: segments.append(bldg_no_seg)
            return segments

        micro_segments = reorder_segments(micro_segments, is_chinese)
        macro_segments = reorder_segments(macro_segments, is_chinese)

        micro_string = "".join("".join(words) for tag, words in micro_segments)
        macro_string = "".join("".join(words) for tag, words in macro_segments)
        
        def clean_string(s):
            s = re.sub(r"^[,/\\\-;\s，。、；]+", "", s)
            s = re.sub(r"[,/\\\-;\s，。、；]+$", "", s)
            return re.sub(r"\s{2,}", " ", s).strip()

        micro_string = clean_string(micro_string)
        macro_string = clean_string(macro_string)
        
        # Calculate Logic Confidence
        split_conf = 1.0
        for k in logic_keys_used:
            split_conf *= conf_mapped.get(k, 1.0)
            
        if not logic_keys_used:
            for k, c in conf_mapped.items():
                if k not in ["district", "region", "sub_district"]: split_conf *= c

        line1 = macro_string if is_chinese else micro_string
        line2 = micro_string if is_chinese else macro_string

        return line1, line2, split_conf, list(logic_keys_used)

    def parse_batch(self, full_addresses, batch_size=32):
        all_results = []
        for i in tqdm(range(0, len(full_addresses), batch_size), desc="Processing"):
            batch = full_addresses[i : i + batch_size]
            
            word_ids, char_ids, masks, batch_tokens = prepare_batch(batch, self.w2i, self.c2i)
            word_ids = word_ids.to(self.device)
            char_ids = char_ids.to(self.device)
            masks = masks.to(self.device)

            with torch.no_grad():
                prediction_ids_batch, emissions_batch = self.model(word_ids, char_ids, masks)
                batch_probs = torch.softmax(emissions_batch, dim=-1).cpu()

            for idx, address_str in enumerate(batch):
                prediction_ids = prediction_ids_batch[idx]
                item_probs = batch_probs[idx]
                tokens = batch_tokens[idx]
                
                # 1. Map tokens back to the original string character by character
                char_tags = ["O"] * len(address_str)
                char_confs = [0.0] * len(address_str)
                
                start_indices = [match.start() for match in re.finditer(r'[a-zA-Z]+|[0-9]+|[\u4e00-\u9fff]|[^\s]', address_str)]
                
                for j, tag_id in enumerate(prediction_ids):
                    if j >= len(start_indices): break
                    start_idx = start_indices[j]
                    token_str = tokens[j]
                    
                    tag = self.idx2tag[tag_id]
                    conf = item_probs[j, tag_id].item()

                    for c in range(start_idx, start_idx + len(token_str)):
                        if char_tags[c] == "O":
                            char_tags[c] = tag
                            char_confs[c] = conf

                # 2. Extract with trailing spaces to preserve formatting
                parsed_entities = []
                for match in re.finditer(r"([a-zA-Z]+|[0-9]+|[\u4e00-\u9fff]|[^\s])(\s*)", address_str):
                    token_str = match.group(1)
                    trailing_space = match.group(2)
                    start_idx = match.start(1)

                    tag = char_tags[start_idx]
                    conf = char_confs[start_idx]
                    entity_group = tag.replace("B-", "").replace("I-", "") if tag != "O" else "O"

                    parsed_entities.append({
                        "entity_group": entity_group,
                        "word": token_str + trailing_space,
                        "conf": conf
                    })

                extracted_raw, conf_raw = self._extract_3d_components(parsed_entities)
                extracted_mapped, conf_mapped = self._map_extracted_to_standard_keys(extracted_raw, conf_raw)
                
                # Split and get logic-aware confidence
                line1, line2, split_conf, used_keys = self._split_address(extracted_mapped, address_str, parsed_entities, conf_mapped)

                all_results.append((address_str, extracted_mapped, conf_mapped, line1, line2, split_conf, used_keys))

        return all_results

# ==========================================
# MAIN EXECUTION & EVALUATION LOOP
# ==========================================
def main():
    if not os.path.exists(MODEL_DIR) or not os.path.exists(TEST_FILE):
        print("❌ Error: Model directory or Test file not found.")
        return

    parser = HKAddressParserBiLSTM(model_path=MODEL_DIR)

    print(f"🚀 Loading dataset from {TEST_FILE}...")
    with open(TEST_FILE, "r", encoding="utf-8") as file:
        test_data = [json.loads(line) for line in file if line.strip() and not line.startswith("#")]
        
    inputs = [item["input"].strip() for item in test_data]
    
    start_time = time.perf_counter()
    results = parser.parse_batch(inputs, batch_size=BATCH_SIZE)
    total_time = time.perf_counter() - start_time

    # Initialize Metrics Structure
    stats = {
        t: {
            "total_in_bin": 0,
            "logic_correct": 0,
            "line1_correct": 0,
            "line2_correct": 0,
            "full_correct": 0
        } for t in THRESHOLDS
    }

    excluded_count = 0

    print(f"✍️ Evaluating and writing results to {LOG_FILE}...")
    with open(LOG_FILE, "w", encoding="utf-8") as log:
        for idx, (address, pred_tags, pred_confs, line1, line2, split_conf, used_keys) in enumerate(results):
            ground_truth_flat = flatten_json(test_data[idx].get("output", {}))
            
            # --- CHECK FOR CORRUPTED GROUND TRUTH ---
            norm_address = normalize_for_eval(address)
            is_corrupted = False
            for _, gt_val in ground_truth_flat.items():
                norm_gt_val = normalize_for_eval(gt_val)
                if norm_gt_val and norm_gt_val not in norm_address:
                    is_corrupted = True
                    break
            
            if is_corrupted:
                excluded_count += 1
                log.write(f"--- Result {idx + 1} [EXCLUDED: CORRUPTED DATA] ---\n")
                log.write(f"Original Input  : {address}\n")
                log.write("Reason          : Ground truth contains values not present in the input text.\n")
                log.write("-" * 50 + "\n")
                continue

            # --- EVALUATE CORRECTNESS OF THE PREDICTION ---
            pred_ff = normalize_for_eval(pred_tags.get("floor", "") + pred_tags.get("flat", ""))
            gt_ff = normalize_for_eval(ground_truth_flat.get("floor", "") + ground_truth_flat.get("flat", ""))
            
            p_bldg = normalize_for_eval(pred_tags.get("building_name", ""))
            p_est  = normalize_for_eval(pred_tags.get("estate_name", ""))
            g_bldg = normalize_for_eval(ground_truth_flat.get("building_name", ""))
            g_est  = normalize_for_eval(ground_truth_flat.get("estate_name", ""))
            
            field_correct = {}
            for field in ALL_FIELDS:
                p_norm = normalize_for_eval(pred_tags.get(field, ""))
                g_norm = normalize_for_eval(ground_truth_flat.get(field, ""))
                
                if p_norm == g_norm:
                    field_correct[field] = True
                elif field in ['floor', 'flat'] and pred_ff == gt_ff and pred_ff != "":
                    field_correct[field] = True
                elif field in ['building_name', 'estate_name'] and (p_bldg == g_est and p_est == g_bldg):
                    field_correct[field] = True
                else:
                    field_correct[field] = False
                    
            # 1. Split Logic Determination Correctness
            is_logic_correct = all(field_correct.get(k, False) for k in used_keys) if used_keys else True
            
            # 2. Line 1 Exact Match (Micro fields)
            line1_fields = ["flat", "floor", "block", "phase", "building_name", "estate_name"]
            is_l1_correct = all(field_correct[f] for f in line1_fields)
            
            # 3. Line 2 Exact Match (Macro fields)
            line2_fields = ["village_name", "building_number", "street_name", "sub_district", "district", "region"]
            is_l2_correct = all(field_correct[f] for f in line2_fields)
            
            # 4. Full Address Exact Match
            is_full_correct = is_l1_correct and is_l2_correct

            # --- BIN THE ACCURACY BY CONFIDENCE ---
            for t in THRESHOLDS:
                if split_conf >= t:
                    stats[t]["total_in_bin"] += 1
                    if is_logic_correct: stats[t]["logic_correct"] += 1
                    if is_l1_correct: stats[t]["line1_correct"] += 1
                    if is_l2_correct: stats[t]["line2_correct"] += 1
                    if is_full_correct: stats[t]["full_correct"] += 1

            # --- WRITE FULL LOGS ---
            log.write(f"--- Result {idx + 1} ---\n")
            log.write(f"Original Input  : {address}\n")
            log.write(f"Logic Keys Used : {used_keys}\n")
            log.write(f"Split Conf      : {split_conf:.6f} (Product of keys used in split)\n")
            log.write(f"Output Line 1   : {line1}\n")
            log.write(f"Output Line 2   : {line2}\n")
            
            if is_full_correct: log.write("✅ EXACT MATCH\n")
            else: log.write("❌ MISMATCH FOUND\n")
            
            for field in ALL_FIELDS:
                pred_val = pred_tags.get(field, "")
                gt_val = ground_truth_flat.get(field, "")
                if pred_val or gt_val:
                    status = "✅" if field_correct[field] else "❌"
                    conf_str = f"  conf={pred_confs.get(field, 0.0):.4f}"
                    log.write(f" {status} {field.upper()}:{conf_str}\n")
                    log.write(f"   PRED: {pred_val if pred_val else '[None]'}\n")
                    log.write(f"   TRUE: {gt_val if gt_val else '[None]'}\n")
                    
            log.write("-" * 50 + "\n")

        # --- WRITE SUMMARY TABLE ---
        total_samples = len(inputs)
        valid_samples = total_samples - excluded_count
        
        def build_table_str():
            out = "\n" + "=" * 65 + "\n"
            out += "📊 ADDRESS SPLITTING METRICS (BiLSTM-CNN-CRF CALIBRATED)\n"
            out += "=" * 65 + "\n"
            out += f"Total Addresses Provided: {total_samples}\n"
            out += f"Excluded (Corrupted)    : {excluded_count}\n"
            out += f"Total Valid Evaluated   : {valid_samples}\n"
            out += f"⏱️ Total Inference runtime: {total_time:.4f} seconds\n\n"

            for t in THRESHOLDS:
                title = "ALL PREDICTIONS (Threshold 0%)" if t == 0.0 else f"PREDICTIONS WITH ≥ {int(t*100)}% CONFIDENCE"
                out += "-" * 65 + "\n"
                out += f"🚀 {title}\n"
                out += "-" * 65 + "\n"
                
                total_in_bin = stats[t]["total_in_bin"]
                if total_in_bin > 0:
                    out += f"{'METRIC':<35} | {'ACCURACY (Correct / Total in Bin)'}\n"
                    out += "-" * 65 + "\n"
                    
                    l_acc = (stats[t]["logic_correct"] / total_in_bin) * 100
                    l1_acc = (stats[t]["line1_correct"] / total_in_bin) * 100
                    l2_acc = (stats[t]["line2_correct"] / total_in_bin) * 100
                    full_acc = (stats[t]["full_correct"] / total_in_bin) * 100
                    
                    out += f"{'Split Logic Determination Correct':<35} | {l_acc:>6.2f}%  ({stats[t]['logic_correct']}/{total_in_bin})\n"
                    out += f"{'Line 1 (Micro) Components Correct':<35} | {l1_acc:>6.2f}%  ({stats[t]['line1_correct']}/{total_in_bin})\n"
                    out += f"{'Line 2 (Macro) Components Correct':<35} | {l2_acc:>6.2f}%  ({stats[t]['line2_correct']}/{total_in_bin})\n"
                    out += f"{'Full Address Perfect Match':<35} | {full_acc:>6.2f}%  ({stats[t]['full_correct']}/{total_in_bin})\n\n"
                else:
                    out += f"No samples met the >= {int(t*100)}% confidence threshold.\n\n"
            return out

        table_output = build_table_str()
        print(table_output)
        log.write(table_output)

if __name__ == "__main__":
    main()

DEBUG: Using Device -> cuda:3
📦 Loading weights from ./bilstm_crf_modelV2/pytorch_model.bin ...
🚀 Loading dataset from data/test.jsonl...


Processing:   0%|          | 0/647 [00:00<?, ?it/s]

✍️ Evaluating and writing results to address_split_results_bilstmV2.log...

📊 ADDRESS SPLITTING METRICS (BiLSTM-CNN-CRF CALIBRATED)
Total Addresses Provided: 20696
Excluded (Corrupted)    : 394
Total Valid Evaluated   : 20302
⏱️ Total Inference runtime: 22.6559 seconds

-----------------------------------------------------------------
🚀 ALL PREDICTIONS (Threshold 0%)
-----------------------------------------------------------------
METRIC                              | ACCURACY (Correct / Total in Bin)
-----------------------------------------------------------------
Split Logic Determination Correct   |  98.34%  (19965/20302)
Line 1 (Micro) Components Correct   |  97.86%  (19868/20302)
Line 2 (Macro) Components Correct   |  93.56%  (18995/20302)
Full Address Perfect Match          |  92.27%  (18733/20302)

-----------------------------------------------------------------
🚀 PREDICTIONS WITH ≥ 20% CONFIDENCE
-----------------------------------------------------------------
METRIC       